# Distribuição Territorial da População

**AUTORIA:** [REDE MOB](https://www.redemob.com.br/)

Este script consolida e apresenta **mapas e estatísticas sociodemográficas** em malha H3 (resolução 9) para os municípios de interesse. Apoia diagnósticos de mobilidade ao espacializar variáveis de população, renda, idade e raça.  

### OBJETIVO

> Construir, a partir de microdados censitários, um conjunto de indicadores espacializados que:
> 1. descrevam a distribuição territorial da população e de suas condições socioeconômicas;  
> 2. sirvam de insumo para análises de equidade, priorização de investimentos e modelos de acessibilidade.

### PANORAMA

- **Leitura de dados:**  
  - Censo Demográfico 2010 e 2022 (IBGE – réplicas no *Base dos Dados*).  
  - Malha H3 derivada dos centroides de setores censitários.  
  - Camada “Área Urbanizada” para delimitação de locais onde há ocupação urbano.  
- **Processamento:** harmonização de dicionários, interseção setor ↦ hexágono (ponderação por área) e cálculo de métricas.  
- **Saídas:** arquivo `parquet` por hexágono, mapas coropléticos (`figures/`) e painéis interativos.

### LIMITAÇÕES

- **Censo 2022:** o Censo apenas liberou dados de rendimento do responsável [até o momento](https://www.ibge.gov.br/novo-portal-destaques/42982-ibge-divulgara-em-30-de-abril-informacoes-do-censo-demografico-2022-sobre-o-rendimento-do-responsavel-pelo-domicilio.html). Com efeito, **não é possível calcular indicadores, como taxa de pobreza e índice de Gini, ou efetuar ranqueamento de divisões territoriais, de acordo com o rendimento**. No entanto, mantivemos a construção de um mapa de população classificado por cores de acordo com o rendimento dos responsáveis por domicílios, pois entendemos que há valor na informação, mas as observações acima devem estar em mente.

### LINKS DE INTERESSE

- IBGE | Censo Demográfico <https://www.ibge.gov.br>  
- Base dos Dados | BigQuery <https://basedosdados.org>  
- Documentação H3 <https://h3geo.org>  


---

## Instruções

Este notebook requer:

1. **Códigos IBGE dos municípios**  
   - Para incluir municípios vizinhos, informe uma lista/tupa de códigos.
2. [**Credenciais de acesso ao BigQuery**](https://basedosdados.org/docs/home#acessando-tabelas-tratadas-bd)
   - Defina a variável de ambiente `GCLOUD_ID` ou insira a _service account key_ correspondente.
3. **Diretório de saída** (`OUTPUT_DIR`)  
   - Caminho onde serão salvos o arquivo `.parquet`, os mapas e demais produtos.  
   - Se omitido, o notebook cria automaticamente a pasta `outputs/` no mesmo diretório do script.


## Definição de Parâmetros

> Na céula a seguir, o usuário deve ajustar os filtros territoriais e temáticos.

| Parâmetro     | Descrição                                   | Exemplo                                   |
|---------------|---------------------------------------------|-------------------------------------------|
| `MUNI_CODES`  | Tupla de códigos IBGE dos municípios-alvo   | `(3300704, 3300258, 3305208)`             |
| `THEMES`      | Blocos temáticos a importar                 | `("basic", "income", "age", "race")`      |
| `H3_RES`      | Resolução da malha H3                       | `9`                                       |
| `OUTPUT_DIR`  | Pasta onde serão salvos os resultados       | `"outputs/"` ou `"/home/usuario/meu_out"` |

```python
# ---- Exemplo 1 --------------------------------------------------------------
GCLOUD_ID   = 'seu-projeto-google-cloud'         # Nome do projeto no Google
MUNI_CODES  = (3300704, 3300258, 3305208)        # Para mais de um município
THEMES      = ("basic", "income", "age", "race") # População, rendimentos, faixas etárias, etnias
H3_RES      = 9                                  # Resolução das "quadrículas"
OUTPUT_DIR  = "outputs"                          # Pasta de "salvamentos"
# -----------------------------------------------------------------------------

# ---- Exemplo 2 --------------------------------------------------------------
GCLOUD_ID   = 'seu-projeto-google-cloud'    # Nome do projeto no Google
MUNI_CODES  = 3300704                       # Para cidade única
THEMES      = ("basic", "age")              # População, faixas etárias
H3_RES      = 7                             # Resolução das "quadrículas"
OUTPUT_DIR  = "outputs"                     # Pasta de "salvamentos"
# -----------------------------------------------------------------------------

## Utilização
1. **Atualize** os parâmetros célula abaixo.  
2. **Execute** todo o notebook. Ele criará a pasta para salvar os resultados armazenará os produtos ali.  
3. **Interprete** os mapas e estatísticas à luz de suas questões de pesquisa ou planejamento.
4. **Role** o notebook para visualizar o passo a passo metodológico em Python: os resultados serão exibidos logo após a seção *backend*. Não é preciso editar o código (não se preocupe), mas ele é mantido para transparência e para que usuários avançados ou curiosos possam manipular procedimentos e compreender a lógica das transformações.

In [ ]:
GCLOUD_ID       = None

MUNI_CODES      = (3106200,)
H3_RESOLUTIONS  = (9, 10, 11)

CRS = 31983

# Backend

## Bibliotecas e Parâmetros Básicos

In [ ]:
# Standard library
import os
from pathlib import Path

# Third-party
import basedosdados as bd
from dotenv import load_dotenv
import geobr
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from tobler.area_weighted import area_interpolate

In [ ]:
if not GCLOUD_ID:
    load_dotenv()
    GCLOUD_ID = os.getenv("GCLOUD_ID")

DATABASE = Path(
    os.environ.get('DB_FOLDER')
    )

OUT_FOLDER = os.environ.get('OUT_FOLDER')
OUT_DIR = Path(OUT_FOLDER) / 'A/sociodemografia'
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
bh = geobr.read_municipality(MUNI_CODES[0]).to_crs(epsg=CRS)


inpath = DATABASE / 'census' / 'grade_estatistica' / 'grade_estatistica_id36.zip'

grid_2010 = (
    gpd
    .read_file(inpath)
    .to_crs(epsg=CRS)
    .pipe(
        lambda gdf: gpd.sjoin(
            gdf,
            bh.reindex(columns=['geometry']),
            how="inner",
            predicate="intersects"
            )
        )
        .drop(columns='index_right')
        .drop_duplicates(subset='ID_UNICO')
    )

In [ ]:
grid_2010.sample(3)

In [ ]:
inpath = DATABASE / 'census' / 'grade_estatistica' / 'grade_id36_2022.zip'

grid_2022 = (
    gpd
    .read_file(inpath)
    .to_crs(epsg=CRS)
    .pipe(
        lambda gdf: gpd.sjoin(
            gdf,
            bh.reindex(columns=['geometry']),
            how="inner",
            predicate="intersects"
            )
        )
        .drop(columns='index_right')
        .drop_duplicates(subset='ID_UNICO')
    )

In [ ]:
grid_2022.sample(3)

In [ ]:
hexes = []
for res in H3_RESOLUTIONS:
    inpath = (
        OUT_DIR
        / f"sociodemografia_hex_r{res}_{'-'.join(str(m) for m in MUNI_CODES)}.parquet"
        )
    hexes.append(
        gpd.read_parquet(inpath).assign(aperture=res)
    )

hexes = pd.concat(hexes)

In [ ]:
hexes = hexes.drop(
    columns=hexes.filter(regex=r"^age_|race_").columns
)

In [ ]:
hexes.sample(2)

In [ ]:
def interpolate_year_population(
    source_grid: gpd.GeoDataFrame,
    target_hexes: gpd.GeoDataFrame,
    *,
    year: int,
    population_column: str,
    equal_area_epsg: int | None = 31983,
    allocate_total: bool = False,
) -> pd.DataFrame:
    """
    Interpolate population from one source grid to target hexagons.

    Parameters
    ----------
    source_grid : GeoDataFrame
        Statistical grid for a single year, with a population column.
    target_hexes : GeoDataFrame
        H3 polygons with a unique hex identifier column.
    year : int
        Reference year to stamp in the output.
    population_column : str
        Column in source_grid with the extensive population variable.
    hex_id_column : str
        Column in target_hexes with unique hex identifiers.
    equal_area_epsg : int, optional
        Equal-area CRS for overlay.
    allocate_total : bool, default False
        If True, preserves the source total by proportional allocation.

    Returns
    -------
    DataFrame with columns [hex_id_column, year, "population"].
    """
    target_crs = equal_area_epsg or target_hexes.crs.to_epsg()

    source = source_grid[[population_column, "geometry"]].to_crs(target_crs)

    target = target_hexes.loc[target_hexes.year == year]
    target = target[["geometry"]].to_crs(target_crs)

    return area_interpolate(
        source_df=source,
        target_df=target,
        extensive_variables=[population_column],
        intensive_variables=None,
        allocate_total=allocate_total,
    )


def interpolate_population_to_h3(
    grid_2010: gpd.GeoDataFrame,
    grid_2022: gpd.GeoDataFrame,
    hex_grid: gpd.GeoDataFrame,
    *,
    population_column_2010: str = "POP",
    population_column_2022: str = "TOTAL",
    equal_area_epsg: int | None = 31983,
    allocate_total: bool = False,
) -> pd.DataFrame:
    """
    Interpolate 2010 and 2022 populations to H3 hexagons and return
    results in long format (hex_id, year, population).
    """
    pop_2010 = interpolate_year_population(
        grid_2010,
        hex_grid,
        year=2010,
        population_column=population_column_2010,
        equal_area_epsg=equal_area_epsg,
        allocate_total=allocate_total,
    ).assign(year=2010)

    pop_2022 = interpolate_year_population(
        grid_2022,
        hex_grid,
        year=2022,
        population_column=population_column_2022,
        equal_area_epsg=equal_area_epsg,
        allocate_total=allocate_total,
    ).assign(year=2022)

    hex_with_grid = pd.concat([pop_2010, pop_2022])

    return (
        hex_with_grid
        .assign(
            habitantes_grid=lambda df: df[population_column_2010].add(df[population_column_2022], fill_value=0)
            )
        .drop(
            columns=[population_column_2010, population_column_2022]
            )
        )


In [ ]:
hexes_with_grid_pop = []
for res, hexes_by_resolution in hexes.groupby("aperture"):
    population_long = interpolate_population_to_h3(
        grid_2010,
        grid_2022,
        hexes_by_resolution,
        population_column_2010="POP",    # grid_2010
        population_column_2022="TOTAL",  # grid_2022
        equal_area_epsg=31983,           # for Brazil, SIRGAS 2000 / UTM 23S
        allocate_total=False,
    )
    hexes_with_grid_pop.append(
        population_long.assign(aperture=res)
        )

hexes_with_grid_pop = pd.concat(hexes_with_grid_pop).set_index('year', append=True)

hexes_with_grid_pop = hexes_with_grid_pop.loc[~hexes_with_grid_pop.index.duplicated()]

In [ ]:
hexes_with_grid_pop.sample(3)

In [ ]:
hexes = hexes.set_index('year', append=True).merge(
    hexes_with_grid_pop[['habitantes_grid']],
    left_index=True,
    right_index=True,
    how='left'
)

In [ ]:
hexes = hexes.round().astype(int, errors='ignore')

In [ ]:
df = hexes
hexes.loc[(hexes.aperture == 10)].xs(key=2022, level='year').plot('habitantes_grid', cmap='magma', scheme='headtailbreaks')

In [ ]:
df = hexes
hexes.loc[(hexes.aperture == 10) & (hexes.habitantes_grid > 1)].xs(key=2010, level='year').plot('b50_households', cmap='magma', scheme='headtailbreaks')

In [ ]:
df_pivot = hexes.reset_index().pivot(index="hex_id", columns="year", values="habitantes_grid")
df_pivot["var_habitantes"] = df_pivot[2022] - df_pivot[2010]

df_result = hexes.merge(
    df_pivot["var_habitantes"],
    left_index=True,
    right_index=True,
    how="left"
).set_index('aperture', append=True)

In [ ]:
df_result.loc[df_result.var_habitantes > 0].xs(key=(2022, 11), level=('year', 'aperture')).plot('var_habitantes', cmap='magma_r', vmax=100)

In [ ]:
df_result.loc[df_result.var_habitantes < 0].xs(key=(2022, 11), level=('year', 'aperture')).plot('var_habitantes', cmap='magma', vmin=-100)

In [ ]:
df_result.to_parquet(
    OUT_DIR / f"sociodemografia.parquet"
)

In [ ]:
df_result.loc[(df_result.filter(like='q').sum(1) - df_result.domicilios) < 100].query("domicilios > 0")

# Exercício: Mapas Coropléticos

Objetivo: Compreender os princípios básicos do mapeamento coroplético e aplicar técnicas de classificação e escolha de cores para representar dados espaciais.

Contexto:

Mapas coropléticos são representações cartográficas que utilizam variações de cor para mostrar valores estatísticos agregados por regiões (como estados, municípios ou bairros). Cada região é associada a um valor e preenchida com uma cor que corresponde à sua classe de valor. Embora hoje seja possível criar mapas sem classificação (unclassed), a abordagem com classes (classed) continua sendo valiosa por facilitar a interpretação visual dos dados.

Três decisões fundamentais no mapeamento coroplético:
- Número de classes: Definir em quantos grupos os valores serão divididos.
- Método de classificação: Escolher o algoritmo para agrupar os dados (ex: quantis, intervalos iguais, Jenks).
- Escolha das cores: Aplicar uma paleta que represente adequadamente as diferenças entre os grupos.

Você deve explorar esses conceitos e, para isso, irá utilizar bibliotecas Python como geopandas e [pacotes do conjunto de bibliotecas PySAL](https://pysal.org/). Acompanhe o capítulo Choropleth Mapping do livro [Geographic Data Science with Python](https://geographicdata.science/book/notebooks/05_choropleth.html) e reproduza o exercício prático lá contido com os municípios acima. Documente o passo a passo e discuta e analise seus achados.